# UQ-SHRED: Neuropixel LFP Dataset

**Data:** Allen Institute Neuropixel Project — Local Field Potential from mouse visual cortex
- Mouse ID: 756029989
- Brain regions: VISam (anteromedial), VISpm (posteromedial), VISp (primary)
- 5 channels per region × 3 regions = **15 total channels**
- 1000 timesteps @ 500 Hz (2 s of drifting grating stimulus)

**Task:** 2 sensors observe → reconstruct all 15 LFP channels with uncertainty

**Experiments:**
- E1: Reconstruction (SHRED vs UQ-SHRED)
- E2: Calibration
- E3: Uncertainty–error relationship
- E4: Per-channel time-series with confidence intervals
- E5: Ablation: number of samples

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler
import os
from datetime import datetime
import json

from processdata import load_data, TimeSeriesDataset
from models import SHRED, UQ_SHRED, fit, fit_uq
import uq

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset_name = 'NEURO'
print(f'Using device: {device}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'results/{dataset_name}_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results: {results_dir}')

np.random.seed(42)
torch.manual_seed(42)

REGION_LABELS = (
    [f'VISam-{i}' for i in range(5)] +
    [f'VISpm-{i}' for i in range(5)] +
    [f'VISp-{i}'  for i in range(5)]
)

## Data — Visualize Raw LFP

In [ ]:
# Quick look at raw data before any preprocessing
raw = np.load('Data/neuro_data.npy', allow_pickle=True).item()
time = raw['time']
visam = raw['visam']  # (5, 1000)
vispm = raw['vispm']
visp  = raw['visp']

region_names = ['VISam', 'VISpm', 'VISp']
region_data  = [visam, vispm, visp]

fig, axes = plt.subplots(3, 5, figsize=(18, 8), sharex=True)
fig.suptitle('Raw LFP — first 1 s (500 samples)', fontsize=14, fontweight='bold')
for ri, (rname, rdata) in enumerate(zip(region_names, region_data)):
    for ch in range(5):
        ax = axes[ri, ch]
        ax.plot(time[:500], rdata[ch, :500], lw=0.8)
        ax.set_title(f'{rname}-ch{ch}', fontsize=8)
        if ri == 2:
            ax.set_xlabel('Time (s)', fontsize=7)
        if ch == 0:
            ax.set_ylabel(rname + '\nV', fontsize=7)
        ax.tick_params(labelsize=6)
plt.tight_layout()
plt.show()

## Data Preparation — 2 sensors, reconstruct 15 channels

In [ ]:
num_sensors = 2
lags = 52   # ~100 ms at 500 Hz

load_X = load_data(dataset_name)  # (1000, 15)
n, m = load_X.shape
print(f'{dataset_name}: {load_X.shape}  ({n} timesteps, {m} channels)')

sensor_locations = np.random.choice(m, size=num_sensors, replace=False)
print(f'Sensor locations: {sensor_locations} = {[REGION_LABELS[i] for i in sensor_locations]}')

In [ ]:
# Time-sequential split: 60% train / 20% valid / 20% test
n_windows = n - lags
n_train = int(0.6 * n_windows)
n_valid = int(0.2 * n_windows)
train_indices = np.arange(0, n_train)
valid_indices = np.arange(n_train, n_train + n_valid)
test_indices  = np.arange(n_train + n_valid, n_windows)

print(f'Windows: {n_windows}  |  Train: {len(train_indices)}, Valid: {len(valid_indices)}, Test: {len(test_indices)}')

sc = MinMaxScaler()
sc.fit(load_X[train_indices])
transformed_X = sc.transform(load_X)

all_data_in = np.zeros((n_windows, lags, num_sensors))
for i in range(n_windows):
    all_data_in[i] = transformed_X[i:i + lags, sensor_locations]

train_data_in  = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
valid_data_in  = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
test_data_in   = torch.tensor(all_data_in[test_indices],  dtype=torch.float32).to(device)
train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
test_data_out  = torch.tensor(transformed_X[test_indices  + lags - 1], dtype=torch.float32).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset  = TimeSeriesDataset(test_data_in,  test_data_out)

print(f'Input shape: {train_data_in.shape}  (batch, lags, sensors)')
print(f'Output shape: {train_data_out.shape}  (batch, channels)')

---
# E1: Reconstruction — SHRED baseline

In [ ]:
shred = SHRED(num_sensors, m, hidden_size=64, hidden_layers=2, l1=128, l2=128, dropout=0.01).to(device)
shred_errors = fit(shred, train_dataset, valid_dataset,
                   batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=20)

shred.eval()
with torch.no_grad():
    shred_recon = shred(test_dataset.X)
    shred_error = (torch.linalg.norm(shred_recon - test_dataset.Y) /
                   torch.linalg.norm(test_dataset.Y)).item()
print(f'SHRED Test Relative Error: {shred_error:.4f}')

# E1: Reconstruction — UQ-SHRED

In [ ]:
uq_shred = UQ_SHRED(num_sensors, m, hidden_size=64, hidden_layers=2,
                    l1=128, l2=128, dropout=0.1, noise_dim=50).to(device)
uq_errors = fit_uq(uq_shred, train_dataset, valid_dataset,
                   batch_size=64, num_epochs=1000, lr=1e-3, verbose=True, patience=20)

In [ ]:
uq_shred.eval()
samples = uq_shred.sample(test_dataset.X, n_samples=100)
samples_np = samples.cpu().numpy()

mean_recon   = samples.mean(dim=0)
median_recon = torch.tensor(np.median(samples_np, axis=0), dtype=torch.float32).to(device)
std_recon    = samples.std(dim=0)

uq_mean_error   = (torch.linalg.norm(mean_recon   - test_dataset.Y) /
                   torch.linalg.norm(test_dataset.Y)).item()
uq_median_error = (torch.linalg.norm(median_recon - test_dataset.Y) /
                   torch.linalg.norm(test_dataset.Y)).item()

shred_recon_np = sc.inverse_transform(shred_recon.cpu().numpy())
test_truth_np  = sc.inverse_transform(test_dataset.Y.cpu().numpy())
samples_orig   = np.array([sc.inverse_transform(s) for s in samples_np])
median_orig    = np.median(samples_orig, axis=0)
std_orig       = samples_orig.std(axis=0)

print(f'UQ-SHRED Mean   Relative Error: {uq_mean_error:.4f}')
print(f'UQ-SHRED Median Relative Error: {uq_median_error:.4f}')

In [ ]:
crps_score = uq.crps(samples, test_dataset.Y)
sharp      = uq.sharpness(samples, conf=0.95)
cal_scores = uq.calibration_scores(samples, test_dataset.Y, levels=[0.5, 0.7, 0.9, 0.95, 0.99])

errors = torch.abs(test_dataset.Y - mean_recon).flatten().cpu().numpy()
stds   = std_recon.flatten().cpu().numpy()
corr   = np.corrcoef(stds, errors)[0, 1]

rmse_shred = np.sqrt(np.mean((test_truth_np - shred_recon_np) ** 2))
rmse_uq    = np.sqrt(np.mean((test_truth_np - median_orig) ** 2))

print('=== E1 Metrics ===')
print(f'SHRED  Rel Error: {shred_error:.4f}   RMSE: {rmse_shred:.8f}')
print(f'UQ Mean   Error:  {uq_mean_error:.4f}')
print(f'UQ Median Error:  {uq_median_error:.4f}   RMSE: {rmse_uq:.8f}')
print(f'CRPS: {crps_score:.6f}')
print(f'Sharpness (95%): {sharp:.6f}')
print(f'Coverage (95%):  {cal_scores[0.95]*100:.1f}%')
print(f'Corr(σ, |e|):    {corr:.4f}')

---
# E2: Calibration

In [ ]:
print('=== E2: Calibration Scores ===')
for level, obs in cal_scores.items():
    print(f'  {level*100:.0f}% CI → {obs*100:.1f}% coverage')

fig, ax = plt.subplots(figsize=(6, 6))
uq.plot_calibration(samples, test_dataset.Y, ax=ax)
ax.set_title('E2: UQ-SHRED Calibration (Neuro)')
plt.tight_layout()
plt.savefig(f'{results_dir}/E2_calibration.png', dpi=150)
plt.show()

---
# E3: Uncertainty–Error Relationship

In [ ]:
idx_sample = np.random.choice(len(errors), min(5000, len(errors)), replace=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stds[idx_sample], errors[idx_sample], alpha=0.2, s=4, color='C0')
z = np.polyfit(stds[idx_sample], errors[idx_sample], 1)
x_line = np.linspace(stds.min(), stds.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'r-', lw=2, label='Linear fit')
ax.set_xlabel('Predicted Uncertainty (σ)', fontsize=12)
ax.set_ylabel('Actual Error |y − ŷ|', fontsize=12)
ax.set_title(f'E3: Uncertainty vs Error  (ρ = {corr:.3f})', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/E3_unc_vs_err.png', dpi=150)
plt.show()

---
# E4: Per-channel time-series with confidence intervals

In [ ]:
q_levels = [0.025, 0.15, 0.25, 0.5, 0.75, 0.85, 0.975]
quantiles_ts = {q: np.percentile(samples_orig, 100 * q, axis=0) for q in q_levels}
T_show = len(test_truth_np)
times_ts = np.arange(T_show)

region_data_split = [('VISam', slice(0, 5)), ('VISpm', slice(5, 10)), ('VISp', slice(10, 15))]

fig, axes = plt.subplots(3, 5, figsize=(20, 9), squeeze=False)
fig.suptitle('E4: UQ-SHRED Reconstruction — All 15 Channels  (2 sensors observed)',
             fontsize=14, fontweight='bold')

for ri, (rname, ch_slice) in enumerate(region_data_split):
    for ch_local in range(5):
        col_idx = ri * 5 + ch_local
        ax = axes[ri, ch_local]
        ax.fill_between(times_ts,
                        quantiles_ts[0.025][:, col_idx],
                        quantiles_ts[0.975][:, col_idx],
                        alpha=0.15, color='C0', label='95% CI')
        ax.fill_between(times_ts,
                        quantiles_ts[0.25][:, col_idx],
                        quantiles_ts[0.75][:, col_idx],
                        alpha=0.35, color='C0', label='50% CI')
        ax.plot(times_ts, test_truth_np[:, col_idx], 'k-', lw=1.2, label='Truth')
        ax.plot(times_ts, shred_recon_np[:, col_idx], 'r-', lw=0.7, alpha=0.7, label='SHRED')
        ax.plot(times_ts, quantiles_ts[0.5][:, col_idx], 'C0--', lw=0.8, label='UQ median')
        ax.set_title(f'{rname}-ch{ch_local}', fontsize=8)
        if ri == 2:
            ax.set_xlabel('time step', fontsize=7)
        if ch_local == 0:
            ax.set_ylabel(f'{rname}\nV', fontsize=7)
        ax.tick_params(labelsize=6)
        if ri == 0 and ch_local == 4:
            ax.legend(fontsize=5, loc='upper right')

plt.tight_layout()
plt.savefig(f'{results_dir}/E4_timeseries_channels.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Region-averaged summary plot
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('E4: Region-Averaged LFP Reconstruction', fontsize=14, fontweight='bold')

for ri, (rname, ch_slice) in enumerate(region_data_split):
    truth_r  = test_truth_np[:, ch_slice].mean(axis=1)
    shred_r  = shred_recon_np[:, ch_slice].mean(axis=1)
    median_r = np.median(samples_orig, axis=0)[:, ch_slice].mean(axis=1)
    lower_r  = np.percentile(samples_orig, 2.5, axis=0)[:, ch_slice].mean(axis=1)
    upper_r  = np.percentile(samples_orig, 97.5, axis=0)[:, ch_slice].mean(axis=1)

    ax = axes[ri]
    ax.fill_between(times_ts, lower_r, upper_r, alpha=0.3, color='C0', label='95% CI')
    ax.plot(times_ts, truth_r,  'k-',   lw=1.5, label='Truth')
    ax.plot(times_ts, shred_r,  'r-',   lw=1.0, alpha=0.8, label='SHRED')
    ax.plot(times_ts, median_r, 'C0--', lw=1.0, label='UQ median')
    ax.set_ylabel(f'{rname}\nV (avg)', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')

axes[-1].set_xlabel('time step')
plt.tight_layout()
plt.savefig(f'{results_dir}/E4_region_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
# E5: Ablation — Number of samples

In [ ]:
n_samples_list = [10, 25, 50, 100, 200]
ablation_results = []

print('=== E5: Ablation Study ===')
for n_samp in n_samples_list:
    samp = uq_shred.sample(test_dataset.X, n_samples=n_samp)
    mean_pred = samp.mean(dim=0)
    rel_error = (torch.linalg.norm(mean_pred - test_dataset.Y) /
                 torch.linalg.norm(test_dataset.Y)).item()
    crps_s = uq.crps(samp, test_dataset.Y)
    sharp_s = uq.sharpness(samp, conf=0.95)
    cal_s = uq.calibration_scores(samp, test_dataset.Y, levels=[0.95])[0.95]
    ablation_results.append({'n_samples': n_samp, 'rel_error': rel_error,
                             'crps': crps_s, 'sharpness': sharp_s, 'coverage_95': cal_s})
    print(f'  n={n_samp:4d}  err={rel_error:.4f}  CRPS={crps_s:.4f}  '
          f'sharp={sharp_s:.4f}  cov={cal_s*100:.1f}%')

In [ ]:
n_arr   = np.array([r['n_samples']  for r in ablation_results])
err_arr = np.array([r['rel_error']  for r in ablation_results])
crps_arr= np.array([r['crps']       for r in ablation_results])
cov_arr = np.array([r['coverage_95']for r in ablation_results]) * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('E5: Ablation — Effect of Sample Count', fontsize=13, fontweight='bold')

axes[0].plot(n_arr, err_arr, 'o-', lw=2, ms=6)
axes[0].axhline(shred_error, color='r', ls='--', label='SHRED')
axes[0].set_xlabel('n_samples'); axes[0].set_ylabel('Relative Error')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(n_arr, crps_arr, 'o-', lw=2, ms=6, color='green')
axes[1].set_xlabel('n_samples'); axes[1].set_ylabel('CRPS')
axes[1].set_title('CRPS'); axes[1].grid(alpha=0.3)

axes[2].plot(n_arr, cov_arr, 'o-', lw=2, ms=6, color='orange')
axes[2].axhline(95, color='k', ls='--', label='Nominal 95%')
axes[2].set_xlabel('n_samples'); axes[2].set_ylabel('Coverage (%)')
axes[2].set_title('Calibration'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/E5_ablation.png', dpi=150)
plt.show()

---
# Summary

In [ ]:
lines = [
    '=' * 70,
    f'UQ-SHRED — {dataset_name}  ({timestamp})',
    '=' * 70,
    f'Sensors observed: {num_sensors}  ({[REGION_LABELS[i] for i in sensor_locations]})',
    f'Channels reconstructed: {m}',
    '',
    f'SHRED  Rel Error: {shred_error:.4f}   RMSE: {rmse_shred:.8f}',
    f'UQ Mean   Error:  {uq_mean_error:.4f}',
    f'UQ Median Error:  {uq_median_error:.4f}   RMSE: {rmse_uq:.8f}',
    '',
    f'CRPS: {crps_score:.6f}',
    f'Sharpness (95%): {sharp:.6f}',
    f'Coverage (95%):  {cal_scores[0.95]*100:.1f}%',
    f'Corr(σ, |e|):    {corr:.4f}',
    '=' * 70,
]
summary = '\n'.join(lines)
print(summary)

with open(f'{results_dir}/metrics.txt', 'w') as f:
    f.write(summary)
print(f'\n✓ Results saved to: {results_dir}/')